In [ ]:
from pathlib import Path
from typing import Literal
import colorsys
from umap import UMAP
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib import colors
import seaborn as sns

from topostats.damage.classes import GrainCollection
from topostats.plottingfuncs import Colormap

CMAP = Colormap().get_cmap()
VMIN = -3
VMAX = 4

In [ ]:
sample_type: Literal["cesium", "proton"] = "cesium"
if sample_type == "cesium":
    dir_base = Path("/Users/sylvi/topo_data/dna_damage_cache")
    grain_collection_file = dir_base / "analysis_results" / "grain_collection.pkl"
    with open(grain_collection_file, "rb") as f:
        grain_collection: GrainCollection = pickle.load(f)
elif sample_type == "proton":
    dir_base = Path("/Users/sylvi/topo_data/dna_damage_proton_cache")
    grain_collection_file = dir_base / "analysis_results" / "grain_collection.pkl"
    with open(grain_collection_file, "rb") as f:
        grain_collection: GrainCollection = pickle.load(f)
else:
    raise ValueError(f"Unknown sample type: {sample_type}")
assert isinstance(grain_collection, GrainCollection)

# plotting
if sample_type == "cesium":
    folder_to_labels = {
        "Controls/nicked": "Control nk",
        "Controls/supercoiled": "Control sc",
        "MilliQ/5_percent_damage": "MilliQ 5%",
        "MilliQ/20_percent_damage": "MilliQ 20%",
        "MilliQ/50_percent_damage": "MilliQ 50%",
        "TE/5_percent_damage": "TE 5%",
        "TE/20_percent_damage": "TE 20%",
        "TE/50_percent_damage": "TE 50%",
    }
    folder_to_colours_hsv = {
        "Control nk": (0 / 360, 0 / 100, 35 / 100),
        "Control sc": (0 / 360, 0 / 100, 70 / 100),
        "MilliQ 5%": (0 / 360, 40 / 100, 40 / 100),
        "MilliQ 20%": (0 / 360, 60 / 100, 60 / 100),
        "MilliQ 50%": (0 / 360, 80 / 100, 80 / 100),
        "TE 5%": (183 / 360, 40 / 100, 40 / 100),
        "TE 20%": (183 / 360, 60 / 100, 60 / 100),
        "TE 50%": (183 / 360, 80 / 100, 80 / 100),
    }
    folder_to_colours_rgb = {folder: colorsys.hsv_to_rgb(*hsv) for folder, hsv in folder_to_colours_hsv.items()}
    plotting_folder_order = [
        "Control nk",
        "Control sc",
        "MilliQ 5%",
        "MilliQ 20%",
        "MilliQ 50%",
        "TE 5%",
        "TE 20%",
        "TE 50%",
    ]
elif sample_type == "proton":
    # - highlet/MQ/100gy: 93
    # - highlet/MQ/50gy: 65
    # - highlet/TE/100gy: 100
    # - highlet/TE/50gy: 60
    # - lowlet/MQ/100gy: 70
    # - lowlet/MQ/50gy: 68
    # - lowlet/TE/100gy: 84
    # - lowlet/TE/50gy: 70
    folder_to_labels = {
        "highlet/MQ/100gy": "High LET MQ 100 Gy",
        "highlet/MQ/50gy": "High LET MQ 50 Gy",
        "highlet/TE/100gy": "High LET TE 100 Gy",
        "highlet/TE/50gy": "High LET TE 50 Gy",
        "lowlet/MQ/100gy": "Low LET MQ 100 Gy",
        "lowlet/MQ/50gy": "Low LET MQ 50 Gy",
        "lowlet/TE/100gy": "Low LET TE 100 Gy",
        "lowlet/TE/50gy": "Low LET TE 50 Gy",
    }
    folder_to_colours_hsv = {
        "High LET MQ 100 Gy": (0 / 360, 40 / 100, 40 / 100),
        "High LET MQ 50 Gy": (0 / 360, 60 / 100, 60 / 100),
        "High LET TE 100 Gy": (183 / 360, 40 / 100, 40 / 100),
        "High LET TE 50 Gy": (183 / 360, 60 / 100, 60 / 100),
        "Low LET MQ 100 Gy": (0 / 360, 80 / 100, 80 / 100),
        "Low LET MQ 50 Gy": (0 / 360, 90 / 100, 90 / 100),
        "Low LET TE 100 Gy": (183 / 360, 80 / 100, 80 / 100),
        "Low LET TE 50 Gy": (183 / 360, 90 / 100, 90 / 100),
    }
    folder_to_colours_rgb = {folder: colorsys.hsv_to_rgb(*hsv) for folder, hsv in folder_to_colours_hsv.items()}
    plotting_folder_order = [
        "High LET MQ 50 Gy",
        "High LET MQ 100 Gy",
        "High LET TE 50 Gy",
        "High LET TE 100 Gy",
        "Low LET MQ 50 Gy",
        "Low LET MQ 100 Gy",
        "Low LET TE 50 Gy",
        "Low LET TE 100 Gy",
    ]

In [ ]:
# load the data
# load the existing analysis results
grain_defect_data_df = pd.read_csv(dir_base / "analysis_results" / "defect_grain_statistics.csv")
print(
    f"Loaded {len(grain_defect_data_df)} rows of defect grain statistics data from {dir_base / 'analysis_results' / 'defect_grain_statistics.csv'}"
)
grain_defect_data_df.head()

# load the tagged data
tagged_data_df = pd.read_csv(dir_base / "analysis_results" / "tagged-output" / "image_tags.csv")
print(
    f"Loaded {len(tagged_data_df)} rows of tagged image statistics data from {dir_base / 'analysis_results' / 'tagged-output' / 'image_tags.csv'}"
)
tagged_data_df.head()
# Get the grain number for each tagged row from the last number in the filename
tagged_data_df["grain_id"] = tagged_data_df["filename"].apply(lambda x: int(x.split("_")[-1].split(".")[0]))

# Merge the two dataframes on the grain_id column
df = pd.merge(tagged_data_df, grain_defect_data_df, on="grain_id", how="inner")

# tag with having any beak if any of the beak columns are True
df["has_any_beak"] = df[["beak_pinch", "beak_full_merge", "beak_partial_merge", "hinge"]].any(axis=1)

# re-save the merged dataframe to a new CSV file
merged_output_path = dir_base / "analysis_results" / "merged_tagged_grain_data.csv"
df.to_csv(merged_output_path, index=False)

In [ ]:
# Plot stuff
# For each column, plot a bar for each of the folder_to_labels, with the height of the bar being the number of true values in that column
cols_to_plot = [
    "has_any_beak",
    "beak_pinch",
    "beak_full_merge",
    "beak_partial_merge",
    "hinge",
    "bad_tracing",
    "double_strand_break",
]
for column in cols_to_plot:
    print(f"plotting {column}")

    counts_per_folder = df.groupby("folder_x")[column].sum()

    # normalise by number of rows in each folder
    counts_per_folder = counts_per_folder / df.groupby("folder_x").size()

    # reorder the counts_per_folder to match the plotting_folder_order
    counts_per_folder = counts_per_folder.reindex(plotting_folder_order)

    print(f"Counts for {column}:")
    print(counts_per_folder)

    # plot bar chart for each folder in the correct order
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x=counts_per_folder.index,
        y=counts_per_folder.values,
    )
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Counts of {column} per folder")
    plt.ylabel(f"Fraction of grains with {column}")
    plt.xlabel("Folder")
    plt.tight_layout()

In [ ]:
# try a umap

# encode the categorical columns as one-hot
sample_type_dummies = pd.get_dummies(df["folder_x"], prefix="folder")
# dummies is a dataframe with columns for each folder, with 1 if the row is in that folder, 0 otherwise

X = pd.concat([sample_type_dummies, df[cols_to_plot]], axis=1).astype(int)

umap_model = UMAP(
    n_neighbors=30,
    min_dist=0.1,
    metric="hamming",
    random_state=0,
)

embedding = umap_model.fit_transform(X)

print(embedding)

In [ ]:
embedding_df = pd.DataFrame(embedding, columns=["umap_1", "umap_2"])
print(embedding_df.head())
embedding_df["folder"] = df["folder_x"].values
embedding_df["grain_id"] = df["grain_id"].values
print(embedding_df.head())

# plot the umap embedding, with points coloured by folder
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x="umap_1",
    y="umap_2",
    hue="folder",
    palette=folder_to_colours_rgb,
    data=embedding_df,
)

In [ ]:
# plot a scatter plot using the images from the grain collection
# iterate over each row in the embedding_df
fig, ax = plt.subplots(figsize=(20, 20))
ax.scatter(embedding_df["umap_1"], embedding_df["umap_2"], alpha=0.5, s=10, c="gray")
for index, row in embedding_df.iterrows():
    grain_id = row["grain_id"]
    folder = row["folder"]
    umap_1 = row["umap_1"]
    umap_2 = row["umap_2"]
    # print(f"plotting grain {grain_id} at ({umap_1}, {umap_2})")

    grain = grain_collection.grains[grain_id]
    grain_image = grain.image

    normalised_colours = colors.Normalize(vmin=VMIN, vmax=VMAX)

    if np.random.rand() > 0.1:
        continue  # only plot 10% of the grains to avoid clutter
    # plot the grain image at the umap_1, umap_2 coordinates
    small_image = OffsetImage(
        grain_image,
        zoom=0.15,
        cmap=CMAP,
        norm=normalised_colours,
    )

    annotation_box = AnnotationBbox(small_image, (umap_1, umap_2), frameon=False, pad=0.0, box_alignment=(0.5, 0.5))
    ax.add_artist(annotation_box)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.imsave(dir_base / "analysis_results" / "umap_embedding.png", fig.canvas.buffer_rgba())
plt.show()